# Kaggriculture v18 public experiment

**Core change:** single-variable market timing: reduce Melon non-terminal sell floor from 0.58 to 0.35

This notebook corresponds to submission/v18/ and records parity, fixed-seed validation, league evidence, error checks, and submission receipts.


In [ ]:
from pathlib import Path
import base64
import io
import json
import sys
import tarfile

ROOT_CANDIDATES = [
    Path.cwd(),
    Path(r"D:\farm\kaggriculture_gold"),
    Path("/kaggle/input/kaggriculture-gold"),
]
ROOT = next((p for p in ROOT_CANDIDATES if (p / "src").exists()), ROOT_CANDIDATES[0])
EMBEDDED_BUNDLE = "..."
if EMBEDDED_BUNDLE:
    ROOT = Path("/kaggle/working")
    with tarfile.open(fileobj=io.BytesIO(base64.b64decode(EMBEDDED_BUNDLE)), mode="r:gz") as bundle:
        bundle.extractall(ROOT)
sys.path.insert(0, str(ROOT / "src" if (ROOT / "src").exists() else ROOT))
print("root:", ROOT)


In [ ]:
from kaggriculture.evaluation.parity import run_phase0
parity = run_phase0(ROOT / "logs" / "v18_parity.json")
parity


In [ ]:
from kaggle_environments import make
from kaggriculture.agents.scenario_agent import agent

def run_episode(seed=2026, steps=720):
    env = make("kaggriculture", configuration={"episodeSteps": steps, "seed": seed}, debug=True)
    env.run([agent, "starter"])
    final = env.steps[-1]
    rewards = [float(state.reward or 0.0) for state in final]
    return {"seed": seed, "rewards": rewards, "winner": "draw" if rewards[0] == rewards[1] else int(rewards[0] > rewards[1])}

result = run_episode()
result


## Reproducibility and publication

- Fixed seeds and the official environment configuration.
- No hidden test labels, private data, or score-dependent branching.
- Promotion decisions use out-of-fold local evidence and league results.
- After Kaggle submission, record the receipt and score in the version manifest.
- A follow-up version changes one approved core variable at a time.
